# M4U3 – Road Pavement Defect Detection: Inference on Unseen Images

This notebook evaluates the trained YOLOv8 road pavement defect detection model on previously unseen road photographs.

The five test images were independently captured by the project author and were not included in the training or validation datasets.

### Detection Classes
1. pothole
2. road_crack
3. uneven_manhole

### Objective
The purpose of this notebook is to examine how the trained model generalizes to new road conditions and to provide visual evidence of successful detections, false positives, and missed detections.

### Workflow
Trained *best.pt* → Unseen Road Images → YOLOv8 Inference → Prediction Evidence

## 2. Environment Setup

Ultralytics is installed and imported in the Google Colab environment. The installed version is recorded to support reproducibility.

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.160


## 3. Load Trained Model

The trained *best.pt* model produced by the reproducible training workflow is stored as a GitHub Release asset. The model is downloaded directly without requiring credentials, Google Drive, or manual file upload.

In [ ]:
import urllib.request
from ultralytics import YOLO

MODEL_URL = "https://github.com/prathish-lab/M4U3-Road-Pavement-Defect-Detection/releases/download/v1.0/best.pt"
MODEL_PATH = "/content/best.pt"

print("Downloading trained model...")
urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

best_model = YOLO(MODEL_PATH)

print("Model loaded successfully.")
print("Classes:", best_model.names)

Model loaded successfully.
Classes: {0: 'pothole', 1: 'road_crack', 2: 'uneven_manhole'}


## 4. Unseen Test Images

Five independently captured road photographs are used to test the model on previously unseen conditions. These images were not included in the training or validation datasets and are stored separately in the GitHub repository for reproducible inference.

In [ ]:
from pathlib import Path
import urllib.request

TEST_DIR = Path("/content/test_images")
TEST_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/prathish-lab/M4U3-Road-Pavement-Defect-Detection/main/test_images"

for i in range(1, 6):
    filename = f"test_{i:02d}.jpg"
    url = f"{BASE_URL}/{filename}"
    destination = TEST_DIR / filename

    urllib.request.urlretrieve(url, destination)
    print(f"Downloaded: {filename}")

print("\nTotal test images:", len(list(TEST_DIR.glob("*.jpg"))))

Downloaded: test_01.jpg
Downloaded: test_02.jpg
Downloaded: test_03.jpg
Downloaded: test_04.jpg
Downloaded: test_05.jpg

Total test images: 5


## 5. Inference on Unseen Images

The trained YOLOv8 model is applied to the five unseen road photographs. Predictions are saved with bounding boxes, class labels, and confidence scores for visual inspection and subsequent error analysis.

In [ ]:
test_images = sorted(TEST_DIR.glob("*.jpg"))

prediction_results = best_model.predict(
    source=[str(img) for img in test_images],
    conf=0.25,
    save=True,
    project="/content/runs",
    name="M4U3_unseen_inference"
)

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (122922240 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


0: 320x640 2 road_cracks, 3.0ms
1: 320x640 1 road_crack, 3.0ms
2: 320x640 (no detections), 3.0ms
3: 320x640 2 uneven_manholes, 3.0ms
4: 320x640 1 road_crack, 3.0ms
Speed: 25.1ms preprocess, 3.0ms inference, 8.3ms postprocess per image at shape (1, 3, 320, 640)
Results saved to /content/runs/M4U3_unseen_inference


## 6. Prediction Evidence

The prediction outputs for the five unseen road photographs are displayed below. These results provide visual evidence of model generalization and are subsequently reviewed for correct detections, false positives, and missed defects.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

prediction_dir = Path("/content/runs/M4U3_unseen_inference")

prediction_images = sorted(prediction_dir.glob("*.jpg"))

print("Prediction images:", len(prediction_images))

for image_path in prediction_images:
    print("\n", image_path.name)
    display(Image(filename=str(image_path), width=700))

## 7. Prediction Summary

The detected classes and confidence scores for each unseen image are summarized below. Images with no detections are also retained because they provide useful evidence for subsequent error analysis.

In [ ]:
for image_path, result in zip(test_images, prediction_results):

    print(f"\n{image_path.name}")

    if len(result.boxes) == 0:
        print("  No detections")
        continue

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        class_name = best_model.names[class_id]

        print(f"  {class_name}: {confidence:.2f}")


test_01.jpg
  road_crack: 0.45
  road_crack: 0.27

test_02.jpg
  road_crack: 0.72

test_03.jpg
  No detections

test_04.jpg
  uneven_manhole: 0.63
  uneven_manhole: 0.49

test_05.jpg
  road_crack: 0.34


## 8. Inference Findings

The unseen-image evaluation demonstrates that the trained YOLOv8 model can identify road-defect features outside the training and validation datasets, but its performance is not consistent across all road conditions.

The model detected *road_crack* and *uneven_manhol*e features in several images. However, some detections corresponded to pavement edges, paving joints, repaired surfaces, or other visually similar features rather than clear defects. One unseen image produced no detections despite visible pavement deterioration.

These observations are consistent with the validation results, where *road_crack* was the most challenging class. The unseen-image results therefore highlight the need for additional diverse training examples and hard-negative samples such as road markings, pavement joints, kerbs, repaired surfaces, and shadows.

The model should be considered an assistive road-screening prototype rather than an autonomous inspection system. Suspected defects require verification by qualified personnel.